# Mini Project 1: Single-Slide Morphology Clustering

Goal: take one public whole-slide image (WSI), tile it, extract morphology features, cluster the tiles, and visualize the result with UMAP and a spatial tile map.

This notebook uses my personal `pathology310` VM environment and LazySlide. It follows the standard LazySlide workflow:

1. Load one public WSI.
2. Detect tissue.
3. Tile tissue regions.
4. Extract deep morphology features from tiles.
5. Run PCA, neighbors, UMAP, and Leiden clustering.
6. Plot UMAP and spatial clusters on the slide.

Beginner note: the full feature extraction step can take time because it runs a vision model over many tiles. The notebook tries to use precomputed LazySlide data first. If features are missing, it runs the full pipeline.

## 0. Environment Check

Run this notebook inside the VM environment:

```bash
source /opt/miniforge3/etc/profile.d/conda.sh
conda activate /opt/miniforge3/envs/pathology310
jupyter lab --no-browser --ip=0.0.0.0 --port=8888
```

In [ ]:
# VM / kernel setup check. Run this cell first.
# If you are using Windows/Antigravity, connect to the VM Jupyter server
# and select the kernel named: Python 3 (pathology310).

import importlib.util
import subprocess
import sys
from getpass import getpass
from pathlib import Path

EXPECTED_ENV = Path("/opt/miniforge3/envs/pathology310")
EXPECTED_PYTHON = EXPECTED_ENV / "bin" / "python"

print("Current notebook Python:", sys.executable)
print("Expected VM Python:", EXPECTED_PYTHON)

if EXPECTED_PYTHON.exists():
    # Register the VM environment as a Jupyter kernel. This is safe to rerun.
    subprocess.run(
        [
            str(EXPECTED_PYTHON),
            "-m",
            "ipykernel",
            "install",
            "--user",
            "--name",
            "pathology310",
            "--display-name",
            "Python 3 (pathology310)",
        ],
        check=False,
    )
else:
    raise SystemExit(
        "This notebook is not running on the VM, or /opt/miniforge3/envs/pathology310 was not found.\n"
        "Start Jupyter on the VM with:\n"
        "  source /opt/miniforge3/etc/profile.d/conda.sh\n"
        "  conda activate /opt/miniforge3/envs/pathology310\n"
        "  jupyter lab --no-browser --ip=0.0.0.0 --port=8888\n"
        "Then open it through your SSH tunnel from Windows."
    )

if not str(sys.executable).startswith(str(EXPECTED_ENV)):
    raise SystemExit(
        "Wrong notebook kernel selected.\n\n"
        f"Current kernel: {sys.executable}\n"
        f"Expected kernel: {EXPECTED_PYTHON}\n\n"
        "Fix in Jupyter/Antigravity:\n"
        "1. Kernel -> Change Kernel\n"
        "2. Select: Python 3 (pathology310)\n"
        "3. Restart kernel\n"
        "4. Run this notebook again from the top."
    )

required_imports = {
    "matplotlib": "matplotlib",
    "scanpy": "scanpy",
    "torch": "torch",
    "lazyslide": "lazyslide",
}
missing = [name for name, module in required_imports.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError(
        "Missing packages in the active pathology310 kernel: " + ", ".join(missing)
    )

import os
from pathlib import Path

import matplotlib.pyplot as plt
import scanpy as sc
import torch

import lazyslide as zs

print("VM notebook setup OK")
print("LazySlide:", zs.__version__)
print("Scanpy:", sc.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Project Settings

`ctranspath` is a pathology-focused vision model supported by LazySlide. It is a good beginner choice for tile feature extraction in this VM because it loads cleanly with the installed package versions.

If feature extraction is slow, reduce the number of tiles by increasing `TILE_PX`, lowering tissue area, or using precomputed data.

Note: `plip` is not the default here because LazySlide `0.9.2` currently passes `use_auth_token` into Transformers `5.3.0`, which raises `TypeError: CLIPModel.__init__() got an unexpected keyword argument 'use_auth_token'` in this VM.

In [ ]:
PROJECT_DIR = Path("single_slide_morphology_outputs")
PROJECT_DIR.mkdir(exist_ok=True)

MODEL_NAME = "ctranspath"
TILE_PX = 256
LEIDEN_RESOLUTION = 0.2
BATCH_SIZE = 32
NUM_WORKERS = 4

feature_table_key = f"{MODEL_NAME}_tiles"
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

if not HF_TOKEN:
    entered_token = getpass("Paste a Hugging Face token if model download requires it, or press Enter to skip: ").strip()
    HF_TOKEN = entered_token or None
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

sc.settings.figdir = str(PROJECT_DIR)
sc.settings.verbosity = 2

print("Output folder:", PROJECT_DIR.resolve())
print("Feature table key:", feature_table_key)
print("HF token available:", bool(HF_TOKEN))

## 2. Load One Public WSI

LazySlide provides a small public GTEx small-intestine WSI dataset through its dataset helper. On first use, it may download data to the Hugging Face cache.

In [ ]:
wsi = zs.datasets.gtex_small_intestine(with_data=True, pbar=True)
wsi

## 3. Tissue Detection And Tiling

This cell prepares tissue regions and tiles. If the public dataset already has saved tissue/tile data, LazySlide may reuse it. If not, it will compute them.

In [ ]:
print("Finding tissue regions...")
zs.pp.find_tissues(wsi)

print("Creating tiles...")
zs.pp.tile_tissues(
    wsi,
    tile_px=TILE_PX,
    background_filter=True,
    background_fraction=0.3,
)

print("Tissue detection and tiling complete.")
wsi

## 4. Visualize Tissue And Tiles

These plots help confirm that tissue detection and tiling worked before spending time on feature extraction.

In [ ]:
zs.pl.tissue(wsi)
plt.show()

zs.pl.tiles(wsi, style="scatter", show_image=True, alpha=0.5, size=2)
plt.show()

## 5. Extract Tile Features

This is the main morphology feature step. Each tile is passed through the selected model and converted into a numeric feature vector.

On the NVIDIA L4 GPU, `amp=True` should usually be faster and use less memory.

In [ ]:
# Hard guard: this VM currently cannot use LazySlide's PLIP loader with Transformers 5.3.0.
# Always force the stable beginner feature extractor before this cell runs.
MODEL_NAME = "ctranspath"
feature_table_key = f"{MODEL_NAME}_tiles"
print("Using feature extractor:", MODEL_NAME)
print("Feature table key:", feature_table_key)

def run_feature_extraction(model_name):
    zs.tl.feature_extraction(
        wsi,
        model=model_name,
        token=HF_TOKEN,
        amp=torch.cuda.is_available(),
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pbar=True,
    )


try:
    adata = wsi[feature_table_key]
    print(f"Found existing feature table: {feature_table_key}")
except Exception:
    print(f"Feature table {feature_table_key!r} not found. Running feature extraction...")
    try:
        run_feature_extraction(MODEL_NAME)
    except TypeError as e:
        if "use_auth_token" not in str(e):
            raise
        print("The selected model hit the LazySlide/Transformers use_auth_token issue.")
        print("Switching feature extractor to ctranspath for this VM.")
        MODEL_NAME = "ctranspath"
        feature_table_key = f"{MODEL_NAME}_tiles"
        run_feature_extraction(MODEL_NAME)

    adata = wsi[feature_table_key]
    print(f"Created feature table: {feature_table_key}")

adata

## 6. PCA, UMAP, And Leiden Clustering

This is the same style of analysis used in single-cell workflows:

- Scale features.
- Run PCA.
- Build a neighbor graph.
- Compute UMAP.
- Cluster with Leiden.

Each point in the UMAP is one WSI tile.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata, random_state=0)
sc.tl.leiden(
    adata,
    flavor="igraph",
    resolution=LEIDEN_RESOLUTION,
    key_added="leiden",
    random_state=0,
)

adata.obs["leiden"].value_counts().sort_index()

## 7. UMAP Plot

Tiles with similar morphology should appear close together. Leiden labels are unsupervised morphology clusters.

In [ ]:
sc.pl.umap(
    adata,
    color="leiden",
    title=f"{MODEL_NAME.upper()} tile morphology clusters",
    save="_single_slide_morphology_clusters.png",
)

print("Saved UMAP to:", PROJECT_DIR / "umap_single_slide_morphology_clusters.png")

## 8. Spatial Cluster Map On The Slide

This maps each tile's cluster back onto the WSI, so you can see where each morphology group appears spatially.

In [ ]:
palette = adata.uns.get("leiden_colors")

zs.pl.tiles(
    wsi,
    feature_key=MODEL_NAME,
    color="leiden",
    alpha=0.65,
    palette=palette,
    show_contours=False,
)
plt.savefig(PROJECT_DIR / "spatial_tile_clusters.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved spatial cluster map to:", PROJECT_DIR / "spatial_tile_clusters.png")

## 9. Optional: LazySlide Spatial Domain Helper

LazySlide also has a helper for spatial domain analysis. This writes domain labels into the tile table, so the plot colors tiles directly by `domain` without passing a feature table.

In [ ]:
zs.tl.spatial_domain(wsi, feature_key=MODEL_NAME, resolution=LEIDEN_RESOLUTION, key_added="domain")

zs.pl.tiles(
    wsi,
    color="domain",
    alpha=0.65,
    show_contours=False,
)
plt.savefig(PROJECT_DIR / "lazy_slide_spatial_domains.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved LazySlide spatial domain map to:", PROJECT_DIR / "lazy_slide_spatial_domains.png")

## 10. Save Results

Save the tile-level metadata with UMAP coordinates and cluster labels.

In [ ]:
obs_path = PROJECT_DIR / "tile_metadata_with_clusters.csv"
adata.obs.to_csv(obs_path)

if "X_umap" in adata.obsm:
    umap_path = PROJECT_DIR / "tile_umap_coordinates.csv"
    umap_df = adata.obs[["leiden"]].copy()
    umap_df["umap_1"] = adata.obsm["X_umap"][:, 0]
    umap_df["umap_2"] = adata.obsm["X_umap"][:, 1]
    umap_df.to_csv(umap_path)
    print("Saved UMAP coordinates to:", umap_path)

print("Saved tile metadata to:", obs_path)

## 11. Optional Advanced Model: CONCH

CONCH means **CONtrastive learning from Captions for Histopathology**. It is a multimodal pathology foundation model from MahmoodLab that can embed pathology image tiles and text prompts into a shared representation space.

Use this section only after the CTransPath workflow works. CONCH is gated on Hugging Face at `MahmoodLab/CONCH`, has a non-commercial license, and requires your Hugging Face account to accept the model terms before download. The VM already has the CONCH Python package available, but model weights still require valid Hugging Face access.

Keep `RUN_CONCH_EXPERIMENT = False` unless you intentionally want to download/run CONCH features. The output feature table will be `conch_tiles`.

In [ ]:
RUN_CONCH_EXPERIMENT = False

if RUN_CONCH_EXPERIMENT:
    if not HF_TOKEN:
        raise ValueError(
            "CONCH is gated on Hugging Face. Set HF_TOKEN after accepting the MahmoodLab/CONCH terms."
        )

    CONCH_MODEL_NAME = "conch"
    conch_feature_table_key = "conch_tiles"

    try:
        conch_adata = wsi[conch_feature_table_key]
        print(f"Found existing feature table: {conch_feature_table_key}")
    except Exception:
        print("Running CONCH feature extraction. This may take longer than CTransPath.")
        zs.tl.feature_extraction(
            wsi,
            model=CONCH_MODEL_NAME,
            token=HF_TOKEN,
            amp=torch.cuda.is_available(),
            batch_size=max(4, BATCH_SIZE // 2),
            num_workers=NUM_WORKERS,
            pbar=True,
        )
        conch_adata = wsi[conch_feature_table_key]

    sc.pp.scale(conch_adata, max_value=10)
    sc.pp.pca(conch_adata, n_comps=50)
    sc.pp.neighbors(conch_adata, n_neighbors=15, n_pcs=30)
    sc.tl.umap(conch_adata, random_state=0)
    sc.tl.leiden(
        conch_adata,
        flavor="igraph",
        resolution=LEIDEN_RESOLUTION,
        key_added="leiden_conch",
        random_state=0,
    )

    sc.pl.umap(
        conch_adata,
        color="leiden_conch",
        title="CONCH tile morphology clusters",
        save="_conch_tile_morphology_clusters.png",
    )

    zs.pl.tiles(
        wsi,
        feature_key=CONCH_MODEL_NAME,
        color="leiden_conch",
        alpha=0.65,
        palette=conch_adata.uns.get("leiden_conch_colors"),
        show_contours=False,
    )
    plt.savefig(PROJECT_DIR / "conch_spatial_tile_clusters.png", dpi=200, bbox_inches="tight")
    plt.show()

    conch_adata.obs.to_csv(PROJECT_DIR / "conch_tile_metadata_with_clusters.csv")
    print("Saved CONCH outputs in:", PROJECT_DIR)
else:
    print("Skipping CONCH. Set RUN_CONCH_EXPERIMENT = True after accepting the HF model terms.")

## What To Try Next

1. Change `LEIDEN_RESOLUTION` to `0.1`, `0.3`, or `0.5` and compare the number of clusters.
2. Change `TILE_PX` from `256` to `128` for finer morphology, but expect more tiles and slower feature extraction.
3. Try CONCH by setting `RUN_CONCH_EXPERIMENT = True` after accepting the `MahmoodLab/CONCH` Hugging Face terms.
4. Try another LazySlide-supported feature model.
5. Select representative tiles from each cluster and inspect them visually.
6. Run the same workflow on another public WSI.